In [ ]:
%%sql -r dataframe_1
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE COMPUTE_WH;
USE DATABASE SUBSCRIPTION_DATA;
USE SCHEMA PROJECT_DATA;

In [ ]:
%%sql -r dataframe_2
SELECT COUNT(*) AS subscriptions
FROM ol_subscriptions;

In [ ]:
%%sql -r dataframe_3
SELECT COUNT(*) AS activity
FROM ol_customer_activity;

In [ ]:
%%sql -r dataframe_4
SELECT COUNT(*) AS demographics
FROM ol_customer_demog;

In [ ]:
%%sql -r dataframe_5
WITH customer_activity AS (

    SELECT
        a.customer_id,
        a.renewed,

        SUM(
            CASE
                WHEN b.activity_date >= '2022-01-01'
                 AND b.activity_date < '2023-01-01'
                THEN b.num_sessions
                ELSE 0
            END
        ) AS total_num_sessions,

        SUM(
            CASE
                WHEN b.activity_date >= '2022-01-01'
                 AND b.activity_date < '2023-01-01'
                THEN b.total_session_length
                ELSE 0
            END
        ) AS gross_total_session_length,

        COUNT(
            CASE
                WHEN b.activity_date >= '2022-01-01'
                 AND b.activity_date < '2023-01-01'
                THEN 1
            END
        ) AS active_days,

        COUNT(
            DISTINCT CASE
                WHEN b.activity_date >= '2022-01-01'
                 AND b.activity_date < '2023-01-01'
                THEN DATE_TRUNC('QUARTER', b.activity_date)
            END
        ) AS active_quarters

    FROM ol_subscriptions a

    LEFT JOIN ol_customer_activity b
        ON a.customer_id = b.customer_id
       AND a.product = b.product

    WHERE
        a.product = 'Healthy Meals'
        AND '2023-01-01' >= a.subscription_start_date
        AND '2023-01-01' < a.subscription_end_date
        AND YEAR(a.subscription_end_date) = 2023

    GROUP BY
        a.customer_id,
        a.renewed
)

SELECT
    c.customer_id,
    c.renewed,
    c.total_num_sessions,
    c.gross_total_session_length,
    c.active_days,
    c.active_quarters,

    ROUND(
        c.total_num_sessions / NULLIF(c.active_quarters, 0),
        2
    ) AS avg_sessions_per_active_quarter,

    d.age,
    d.education,
    d.income_level,
    d.device_type,
    d.tech_comfort_score

FROM customer_activity c

LEFT JOIN ol_customer_demog d
    ON c.customer_id = d.customer_id

ORDER BY c.customer_id;

In [ ]:
# Convert the SQL results to a pandas DataFrame
df = dataframe_5.to_pandas()

# Check that it loaded correctly
print(df.shape)
df.head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Make a copy
model_df = df.copy()

# Encode categorical columns
label_encoders = {}

categorical_columns = [
    "EDUCATION",
    "INCOME_LEVEL",
    "DEVICE_TYPE"
]

for col in categorical_columns:
    le = LabelEncoder()
    model_df[col] = le.fit_transform(model_df[col].astype(str))
    label_encoders[col] = le

# Features
X = model_df.drop(columns=["CUSTOMER_ID", "RENEWED"])

# Target
y = model_df["RENEWED"]

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training rows:", X_train.shape)
print("Testing rows:", X_test.shape)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Create the model
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    max_depth=15
)

# Train the model
rf_model.fit(X_train, y_train)

print("Random Forest model trained successfully! Woot!")

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Make predictions
y_pred = rf_model.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

# Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# Feature Importance
import pandas as pd

feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

feature_importance

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Shared palette - same colors, same order, used identically in every plot
PALETTE = ['#2196F3', '#FF9800', '#4CAF50', '#F44336', '#9C27B0', '#009688', '#FFC107']


# Plot: Feature Importance (Random Forest)
fig, ax = plt.subplots(figsize=(9, 7))

# Plot top-to-bottom with the most important feature at the top
fi_plot = feature_importance.sort_values(by="Importance", ascending=True)
n = len(fi_plot)

ax.barh(fi_plot['Feature'], fi_plot['Importance'],
        color=[PALETTE[i % len(PALETTE)] for i in range(n)])
ax.set_xlabel('Importance')
ax.set_title('Feature Importance (Random Forest)')

# Label each bar with its value
for i, (feature, importance) in enumerate(zip(fi_plot['Feature'], fi_plot['Importance'])):
    ax.annotate(f'{importance:.2f}', xy=(importance, i),
                xytext=(4, 0), textcoords='offset points',
                va='center', fontsize=7)

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# STEP 1 - Persist the trained model

import pickle

model_path = "/tmp/capstone_rf_model_v1.pkl"
encoder_path = "/tmp/capstone_label_encoders_v1.pkl"

with open(model_path, "wb") as f:
    pickle.dump(rf_model, f)

with open(encoder_path, "wb") as f:
    pickle.dump(label_encoders, f)

print(f"Model saved: {model_path}")
print(f"Encoder saved: {encoder_path}")

In [ ]:
# Load the saved model and encoders

with open(model_path, "rb") as f:
    loaded_model = pickle.load(f)

with open(encoder_path, "rb") as f:
    loaded_encoders = pickle.load(f)

print("Model and encoders loaded successfully!")
print(type(loaded_model))
print(type(loaded_encoders))

In [ ]:
# Make predictions on the test data using the loaded model
loaded_predictions = loaded_model.predict(X_test)

print("First 10 predictions:")
print(loaded_predictions[:10])

In [ ]:
comparison = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": loaded_predictions
})

comparison["Correct"] = comparison["Actual"] == comparison["Predicted"]

print(comparison.head(20))

In [ ]:
%%sql -r dataframe_6
SHOW STAGES IN SCHEMA PROJECT_DATA;

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

print(session)

In [ ]:
# STEP 2 - Upload model artifacts to the Snowflake stage

for local_path in [model_path, encoder_path]:
    result = session.file.put(
        local_path,
        "@SUBSCRIPTION_DATA.PROJECT_DATA.CHURN_MODEL_STAGE",
        overwrite=True,
        auto_compress=False
    )

    print(f"Uploaded: {local_path} -> {result[0].target}")

print("\nAll artifacts uploaded.")
print("Model is available in:")
print("@SUBSCRIPTION_DATA.PROJECT_DATA.CHURN_MODEL_STAGE")

Random Forest Model Summary

This notebook builds and evaluates a Random Forest model to predict whether a customer is likely to renew their subscription. The data was first prepared and preprocessed before being split into training and testing datasets. A Random Forest classifier was then trained and evaluated using accuracy, a confusion matrix, and a classification report.

After training, the model's feature importance was reviewed to better understand which variables had the greatest impact on the predictions. The trained model and label encoders were then saved, reloaded, and tested to confirm they could successfully generate predictions on the test dataset.

The following files were created for use in the Streamlit application:

capstone_rf_model_v1.pkl – Trained Random Forest model
capstone_label_encoders_v1.pkl – Saved label encoders used to preprocess categorical variables

These files will be used by the Streamlit application to generate predictions for new customer data.